# Hydrological Phase Analysis of Gauge Series

Classifies every day of one or more **daily water-level (stage) series** into a
hydrological phase — **Low, Rising, High or Falling** — and produces
publication-ready figures and tables from the result.

## What this notebook does

For a folder of gauge CSV files (one file per station) and an optional station
metadata table, it produces:

1. **Phase tables** — one CSV per station with `date, level, phase`.
2. **Figure 1 — river panels** — one figure per river; each station shows the
   complete series coloured by phase, a monthly phase heat-map, the recent
   period and the seasonality (monthly percentiles).
3. **Figure 2 — seasonality** — one compact figure with the seasonal cycle of
   every station, grouped by river, with each month coloured by its seasonal
   phase.
4. **Figure 3 — station summary cards** — period of record, mean annual
   amplitude, number of observations and data gaps for every station, plus a
   CSV with the same numbers.

## How the phases are defined

1. Gaps of up to `gap_days` days (default 15) are linearly interpolated; longer
   gaps stay empty.
2. The **hydrological year** starts in the calendar month with the lowest mean
   level.
3. Within each hydrological year, the 25th and 75th percentiles of the level
   are the *Low* and *High* thresholds.
4. The level is smoothed with a centred 30-day rolling mean (the *trend*). Each
   day is then labelled, in this order: trend ≥ high threshold → **High**;
   trend ≤ low threshold → **Low**; trend higher than the previous day →
   **Rising**; otherwise → **Falling**.

## Input

- `input_dir`: one CSV per station named `<code>.csv`, with a date column and a
  water-level column (names configurable in `Config`). Paths are resolved
  from the repository root.
- `metadata_path` *(optional)*: one row per station with its code, river and a
  short region tag. Without it all stations are grouped as "Unknown" and
  Figures 2 and 3 are skipped.

## How to run

Edit `CFG` in **Block 0** (paths and column names), then *Run All*. Each output
can be switched off with the `RUN_*` flags in **Block 7**. Everything is
written under `output_dir` (`output/` by default).


In [ ]:
# =============================================================================
# BLOCK 0 — IMPORTS AND CONFIGURATION
# =============================================================================
# Everything a user normally needs to change lives in this block: paths, input
# column names and analysis parameters are all fields of `Config`; the rest of
# the notebook only reads from it.

import math
import textwrap
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

# Resolve paths from the repository root, whether Jupyter starts there or
# inside notebooks/hydrological_phase_classification.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'environment.yml').exists() and (p / 'notebooks').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from inside a checkout of the repository.')

# -----------------------------------------------------------------------------
# Hydrological phases (fixed vocabulary used by every table and figure)
# -----------------------------------------------------------------------------
# The order below is meaningful: it drives legend order, heat-map colour codes
# and marker drawing order.
PHASES = ("Low", "Rising", "High", "Falling")

# Okabe-Ito colour-blind-safe palette.
PHASE_COLORS = {
    "Low": "#D55E00",      # vermillion
    "Rising": "#009E73",   # bluish green
    "High": "#0072B2",     # blue
    "Falling": "#E69F00",  # orange
}


# -----------------------------------------------------------------------------
# Figure 1 layout presets
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class Tier:
    """Layout preset for one river's figure (one block per station)."""
    row_height: float      # height of one station block, in inches
    font: float            # tick-label font size
    panel_font: float      # panel-header font size
    marker_ts: float       # marker size, complete time series
    marker_recent: float   # marker size, recent-period panel
    heatmap_years: int     # number of years shown in the phase heat-map
    row_hspace: float      # vertical gap between station blocks
    bottom_ratio: float    # height of the lower panel row relative to the upper


# Preset chosen from the number of stations on the river:
# (largest station count that uses the preset, preset).
STATIONS_COUNT_TIERS: List[Tuple[float, Tier]] = [
    (1, Tier(3.25, 7.4, 8.0, 4.00, 7.0, 8, 0.180, 0.48)),
    (2, Tier(2.55, 6.8, 7.5, 3.20, 6.0, 7, 0.160, 0.42)),
    (3, Tier(2.15, 6.3, 7.0, 2.60, 5.2, 7, 0.140, 0.40)),
    (7, Tier(1.62, 5.6, 6.3, 1.90, 4.0, 6, 0.105, 0.34)),
    (math.inf, Tier(1.32, 5.0, 5.8, 1.35, 3.2, 5, 0.075, 0.30)),
]

# Optional per-river presets (first match wins). Keys are lower-case substrings
# of the river name. These hand-tuned values reproduce the Pantanal (Brazil)
# figures; pass `river_style_overrides=()` to use only the station-count presets.
PANTANAL_RIVER_OVERRIDES: Tuple[Tuple[Tuple[str, ...], Tier], ...] = (
    (("paraguay",),           Tier(1.28, 4.90, 5.6, 1.25, 3.0, 5, 0.065, 0.28)),
    (("cuiaba", "cuiabá"),    Tier(1.42, 5.15, 5.9, 1.45, 3.3, 5, 0.075, 0.30)),
    (("taquari",),            Tier(3.05, 7.20, 7.8, 3.80, 6.8, 8, 0.160, 0.46)),
    (("jauru",),              Tier(2.50, 6.80, 7.5, 3.00, 5.8, 7, 0.150, 0.42)),
    (("piquiri",),            Tier(2.48, 6.80, 7.5, 3.00, 5.8, 7, 0.150, 0.42)),
    (("aquida",),             Tier(2.45, 6.70, 7.4, 2.80, 5.6, 7, 0.150, 0.42)),
    (("lourenco", "lourenço"), Tier(2.05, 6.20, 6.9, 2.40, 5.0, 6, 0.130, 0.38)),
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Config:
    """All user-facing settings. Only override what differs from the defaults."""

    # ---- Input files ---------------------------------------------------------
    input_dir: Path = REPO_ROOT / "input" / "gauge_series"  # one CSV per station: <code>.csv
    metadata_path: Optional[Path] = REPO_ROOT / "input" / "station_metadata.csv"

    # ---- Input column names --------------------------------------------------
    date_col: str = "date"
    # Used only when `date_col` is absent: (day, month, year) column names.
    date_parts_cols: Optional[Tuple[str, str, str]] = None
    # Water-level column: the first name found in a file is used.
    level_cols: Tuple[str, ...] = ("level", "stage", "cota", "Cota", "wse")
    # Metadata columns (metadata is optional; see `load_metadata`).
    meta_code_col: str = "code"      # station code, must match the CSV file name
    meta_river_col: str = "river"    # used to group stations into river figures
    meta_region_col: str = "region"  # short tag shown next to the station code
    meta_name_col: str = "name"      # station name (informational)

    # ---- Outputs -------------------------------------------------------------
    output_dir: Path = REPO_ROOT / "output"
    # Where the per-station phase tables are written. Point this at `input_dir`
    # to overwrite the input CSVs in place (they keep only the three columns
    # listed in `phase_table_cols`).
    phase_tables_dir: Path = REPO_ROOT / "output" / "phase_tables"
    phase_table_cols: Tuple[str, str, str] = ("date", "level", "phase")
    dataset_label: str = "gauges"    # used in the river-panel file names
    show_figures: bool = True        # display figures inline (they are always saved)

    # ---- Phase classification ------------------------------------------------
    gap_days: int = 15               # gaps up to this length are linearly filled;
                                     # longer gaps are reported as "large gaps"
    trend_window_days: int = 30      # centred rolling-mean window (days)
    low_quantile: float = 0.25       # level <= this quantile of the hydro-year -> Low
    high_quantile: float = 0.75      # level >= this quantile of the hydro-year -> High
    hydro_year_start_month: Optional[int] = None  # None: month with the lowest mean level

    # ---- Figures -------------------------------------------------------------
    recent_start: str = "2023-01-01"        # start of the "recent period" panels
    river_panel_width_in: float = 11.69     # A4 width
    river_style_overrides: Tuple = PANTANAL_RIVER_OVERRIDES
    seasonality_max_cols: int = 3
    station_cards_max_cols: int = 3


# -----------------------------------------------------------------------------
# YOUR SETTINGS
# -----------------------------------------------------------------------------
CFG = Config(
    input_dir=REPO_ROOT / "input" / "gauge_series",
    metadata_path=REPO_ROOT / "input" / "station_metadata.csv",
    output_dir=REPO_ROOT / "output",
    phase_tables_dir=REPO_ROOT / "output" / "phase_tables",
)

# Example — settings that reproduce the original Brazilian ANA / Pantanal run
# (Portuguese column names; phase tables overwrite the input CSVs in place):
#
# CFG = Config(
#     input_dir=Path("dados_processados/ana/sem_nivelamento"),
#     metadata_path=Path("analises/tabelas/ana_metadados.csv"),
#     date_col="data",
#     meta_code_col="Código", meta_river_col="Rio",
#     meta_region_col="Pantanal", meta_name_col="Nome",
#     output_dir=Path("figuras"),
#     phase_tables_dir=Path("dados_processados/ana/sem_nivelamento"),
#     phase_table_cols=("data", "wse", "phase"),
#     dataset_label="sem_nivelamento",
# )


In [ ]:
# =============================================================================
# BLOCK 1 — DATA LOADING
# =============================================================================
# One station = one CSV file (<code>.csv) with a date column and a water-level
# column. An optional metadata table describes the stations (river, region...).
# Series are cleaned once here and reused by every later block.


@dataclass
class Station:
    """A gauge series together with the metadata needed for the figures."""
    code: str
    river: str = "Unknown"
    region: str = ""
    name: str = ""
    series: pd.DataFrame = field(default_factory=pd.DataFrame)  # columns: date, level[, phase]


def clean_text(value, default: str = "") -> str:
    """Stripped string; NaN, 'nan' and blank values become `default`."""
    text = "" if value is None else str(value).strip()
    return default if text == "" or text.lower() == "nan" else text


def safe_filename(value) -> str:
    """Turn any label into a string that is safe to use in a file name."""
    text = str(value).strip() or "unknown"
    text = "".join(c if c.isalnum() or c in " -_" else "_" for c in text)
    return text.replace(" ", "_")


def load_metadata(cfg: Config) -> pd.DataFrame:
    """Station metadata (all columns as read). Empty frame if not available."""
    path = cfg.metadata_path
    if path is None or not Path(path).exists():
        print(f"Metadata file not found ({path}); stations will be grouped as 'Unknown'.")
        return pd.DataFrame()
    meta = pd.read_csv(path)
    if cfg.meta_code_col not in meta.columns:
        raise ValueError(f"Metadata has no '{cfg.meta_code_col}' column (see Config.meta_code_col).")
    meta[cfg.meta_code_col] = meta[cfg.meta_code_col].astype(str)
    return meta


def station_from_metadata(code: str, meta: pd.DataFrame, cfg: Config, series=None) -> Station:
    """Build a `Station`, taking river/region/name from the first matching metadata row."""
    row = None
    if not meta.empty:
        match = meta[meta[cfg.meta_code_col] == code]
        if not match.empty:
            row = match.iloc[0]

    def pick(column: str, default: str = "") -> str:
        value = row[column] if row is not None and column in row.index else None
        return clean_text(value, default)

    return Station(
        code=code,
        river=pick(cfg.meta_river_col, "Unknown"),
        region=pick(cfg.meta_region_col),
        name=pick(cfg.meta_name_col),
        series=series if series is not None else pd.DataFrame(),
    )


def load_station_series(path: Path, cfg: Config) -> pd.DataFrame:
    """Read one gauge CSV and return a clean series with columns [date, level].

    Rows with an unparseable date or level are dropped, duplicated dates keep
    their first occurrence and the result is sorted by date.
    """
    raw = pd.read_csv(path)

    # Date: a single column, or (day, month, year) columns.
    if cfg.date_col in raw.columns:
        date = pd.to_datetime(raw[cfg.date_col], errors="coerce")
    elif cfg.date_parts_cols and all(c in raw.columns for c in cfg.date_parts_cols):
        day, month, year = (pd.to_numeric(raw[c], errors="coerce") for c in cfg.date_parts_cols)
        date = pd.to_datetime(pd.DataFrame({"year": year, "month": month, "day": day}), errors="coerce")
    else:
        raise ValueError(f"no date column ('{cfg.date_col}') found")

    # Water level: first candidate column present in the file.
    level_col = next((c for c in cfg.level_cols if c in raw.columns), None)
    if level_col is None:
        raise ValueError(f"no water-level column found (looked for {list(cfg.level_cols)})")
    level = pd.to_numeric(raw[level_col], errors="coerce")

    series = pd.DataFrame({"date": date, "level": level})
    series = series.sort_values("date", kind="mergesort").dropna().drop_duplicates("date")
    if series.empty:
        raise ValueError("no valid data after cleaning")
    return series.reset_index(drop=True)


def load_all_series(cfg: Config) -> Dict[str, pd.DataFrame]:
    """Load every `<code>.csv` in `cfg.input_dir`; stations that fail are skipped."""
    input_dir = Path(cfg.input_dir)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    files = sorted(input_dir.glob("*.csv"))
    print(f"Loading {len(files)} station file(s) from {input_dir} ...")
    series_by_code = {}
    for path in files:
        try:
            series_by_code[path.stem] = load_station_series(path, cfg)
        except Exception as exc:  # keep going: one bad file must not stop the batch
            print(f"  skipped {path.stem}: {exc}")
    return series_by_code


In [ ]:
# =============================================================================
# BLOCK 2 — HYDROLOGICAL PHASE CLASSIFICATION
# =============================================================================
# Every day of a station's record is labelled Low / Rising / High / Falling:
#
#   1. Gaps of up to `gap_days` days are linearly interpolated, so the series is
#      continuous (longer gaps stay empty).
#   2. The hydrological year starts in the calendar month with the lowest mean
#      level, so each flood pulse falls inside a single hydrological year.
#   3. Per hydrological year, two thresholds are computed from the level
#      distribution: the `low_quantile` and the `high_quantile`.
#   4. The level is smoothed with a centred `trend_window_days`-day rolling mean
#      ("trend"), and each day is classified in this order:
#         trend >= high threshold                -> High
#         trend <= low  threshold                -> Low
#         otherwise, trend above previous day's  -> Rising
#         otherwise                              -> Falling


def interpolate_short_gaps(series: pd.DataFrame, gap_days: int) -> pd.Series:
    """Level on a continuous daily calendar, with gaps <= `gap_days` days filled."""
    calendar = pd.date_range(series["date"].min(), series["date"].max())
    level = series.set_index("date")["level"].reindex(calendar).astype(float)
    return level.interpolate(method="linear", limit=gap_days)


def classify_phases(level: pd.Series, cfg: Config) -> pd.Series:
    """Phase (Low/Rising/High/Falling) for every day of a gap-filled daily level series."""
    months, years = level.index.month, level.index.year

    # 1. Hydrological year: starts in the month with the lowest mean level.
    monthly_mean = level.groupby(months).mean()
    start_month = cfg.hydro_year_start_month or (10 if monthly_mean.empty else monthly_mean.idxmin())
    hydro_year = np.where(months >= start_month, years, years - 1)

    # 2. Low / High thresholds of each hydrological year, mapped back to every day.
    thresholds = level.groupby(hydro_year).agg(
        q_low=lambda x: x.quantile(cfg.low_quantile),
        q_high=lambda x: x.quantile(cfg.high_quantile),
    )
    q_low = pd.Series(hydro_year, index=level.index).map(thresholds["q_low"])
    q_high = pd.Series(hydro_year, index=level.index).map(thresholds["q_high"])

    # 3. Smoothed trend and its day-to-day change.
    trend = level.rolling(window=cfg.trend_window_days, center=True, min_periods=1).mean()
    is_rising = trend.diff().fillna(0) > 0

    # 4. Rules applied in priority order (comparisons with NaN are False, so days
    #    inside a long unfilled gap fall through to "Falling").
    phase = np.select(
        [trend >= q_high, trend <= q_low, is_rising],
        ["High", "Low", "Rising"],
        default="Falling",
    )
    return pd.Series(phase, index=level.index, name="phase")


def add_phase(series: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Copy of a clean [date, level] series with a `phase` column for each observation."""
    daily_phase = classify_phases(interpolate_short_gaps(series, cfg.gap_days), cfg)
    out = series.copy()
    out["phase"] = daily_phase.reindex(out["date"]).to_numpy()
    out["phase"] = out["phase"].fillna("Undefined")
    return out


def write_phase_tables(stations: List[Station], cfg: Config) -> None:
    """Write one `<code>.csv` per station with the columns `cfg.phase_table_cols`."""
    out_dir = Path(cfg.phase_tables_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    date_name, level_name, phase_name = cfg.phase_table_cols
    for st in stations:
        table = pd.DataFrame({
            date_name: st.series["date"],
            level_name: st.series["level"],
            phase_name: st.series["phase"],
        })
        table.to_csv(out_dir / f"{st.code}.csv", index=False)
    print(f"Wrote {len(stations)} phase table(s) to {out_dir}")


In [ ]:
# =============================================================================
# BLOCK 3 — STATION STATISTICS
# =============================================================================
# Small, plot-independent helpers shared by the figures: monthly percentiles,
# month-to-phase mapping (seasonality), station ordering and the summary values
# printed on the station cards.


def monthly_percentiles(series: pd.DataFrame) -> pd.DataFrame:
    """P10/P25/P50/P75/P90 of the water level for each calendar month (index = month)."""
    valid = series.dropna(subset=["level"])
    stats = valid.groupby(valid["date"].dt.month)["level"].agg(
        p10=lambda x: x.quantile(0.10),
        p25=lambda x: x.quantile(0.25),
        p50="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
    )
    stats.index.name = "month"
    return stats


def peak_month(monthly: pd.DataFrame) -> int:
    """Calendar month with the highest median level (1 if there is no data)."""
    return 1 if monthly.empty else int(monthly["p50"].idxmax())


def _circular(month: int) -> int:
    """Wrap any integer into the 1-12 month range."""
    return (month - 1) % 12 + 1


def assign_month_phases(monthly: pd.DataFrame) -> Dict[int, str]:
    """Give each calendar month a phase, with exactly three months per phase.

    The three consecutive months with the highest summed median level (windows
    that contain the peak month) form the High season; the cycle then continues
    High -> Falling -> Low -> Rising. Series with fewer than 12 months of
    statistics fall back to a fixed January-start cycle.
    """
    if len(monthly) < 12:
        fallback_cycle = ["Rising"] * 3 + ["High"] * 3 + ["Falling"] * 3 + ["Low"] * 3
        return {month: fallback_cycle[month - 1] for month in range(1, 13)}

    medians = monthly["p50"]
    peak = peak_month(monthly)
    candidate_windows = [
        [_circular(peak - 2), _circular(peak - 1), peak],   # peak closes the window
        [_circular(peak - 1), peak, _circular(peak + 1)],   # peak in the middle
        [peak, _circular(peak + 1), _circular(peak + 2)],   # peak opens the window
    ]

    def window_total(w: List[int]) -> float:
        # Plain left-to-right addition on purpose: the built-in `sum()` switches to
        # compensated summation for Python floats on Python >= 3.12, which would
        # break exact ties between windows differently across Python versions.
        return medians[w[0]] + medians[w[1]] + medians[w[2]]

    best_window = max(candidate_windows, key=window_total)   # first window wins ties
    first_high_month = best_window[0]

    cycle = ["High"] * 3 + ["Falling"] * 3 + ["Low"] * 3 + ["Rising"] * 3
    return {_circular(first_high_month + i): phase for i, phase in enumerate(cycle)}


def sort_by_peak_month(stations: List[Station]) -> List[Station]:
    """Stations ordered by the month of their seasonal peak (ties keep input order)."""
    return sorted(stations, key=lambda st: peak_month(monthly_percentiles(st.series)))


def group_by_river(stations: List[Station]) -> Dict[str, List[Station]]:
    """{river name: stations}, rivers sorted alphabetically."""
    rivers: Dict[str, List[Station]] = {}
    for st in stations:
        rivers.setdefault(st.river, []).append(st)
    return dict(sorted(rivers.items()))


def station_summary(st: Station, cfg: Config) -> dict:
    """Numbers shown on a station card: period, amplitude, observations and gaps."""
    series = st.series
    year = series["date"].dt.year
    yearly = series.groupby(year)["level"]
    amplitude = (yearly.max() - yearly.min()).mean()

    # Gaps longer than `gap_days` between consecutive observations.
    step = series["date"].diff()
    is_gap = step > pd.Timedelta(days=cfg.gap_days)
    gap_days = step[is_gap].dt.days
    if gap_days.empty:
        longest_gap = "None"
    else:
        end = series.loc[gap_days.idxmax(), "date"]   # first longest gap wins ties
        start = series["date"].shift().loc[gap_days.idxmax()]
        longest_gap = f"{int(gap_days.max())} d ({start:%Y-%m-%d} to {end:%Y-%m-%d})"

    return {
        "river": st.river,
        "code": st.code,
        "name": st.name,
        cfg.meta_region_col.lower(): st.region,
        "period": f"{series['date'].min():%Y-%m-%d} to {series['date'].max():%Y-%m-%d}",
        "mean_annual_amplitude": float(amplitude) if pd.notna(amplitude) else np.nan,
        "large_gaps": int(is_gap.sum()),
        "longest_gap": longest_gap,
        "n_obs": int(len(series)),
    }


In [ ]:
# =============================================================================
# BLOCK 4 — FIGURE 1: STATION PANELS, ONE FIGURE PER RIVER
# =============================================================================
# Each station is a block of four panels:
#     a) complete series + phase heat-map        b) recent period + seasonality
# Blocks are stacked vertically, one figure per river, sized to fit an A4 page.

MONTH_LETTERS = list("JFMAMJJASOND")
MONTH_LETTERS_SPARSE = ["J", "", "M", "", "M", "", "J", "", "S", "", "N", ""]  # for tiny fonts


@dataclass(frozen=True)
class RiverLayout:
    """Every size used to draw one river figure (derived from a `Tier`)."""
    fig_width: float
    fig_height: float
    top: float
    main_title_font: float
    panel_font: float
    font: float
    label_font: float
    legend_font: float
    legend_marker: float
    marker_ts: float
    marker_recent: float
    line_width: float
    heatmap_years: int
    row_hspace: float
    inner_hspace: float
    inner_wspace: float
    bottom_ratio: float


def river_layout(river_name: str, n_stations: int, cfg: Config) -> RiverLayout:
    """Choose sizes from the station count, then apply any per-river override."""
    tier = next(t for max_n, t in STATIONS_COUNT_TIERS if n_stations <= max_n)
    key = str(river_name).lower()
    for keywords, override in cfg.river_style_overrides:
        if any(k in key for k in keywords):
            tier = override
            break

    return RiverLayout(
        fig_width=cfg.river_panel_width_in,
        fig_height=0.95 + n_stations * tier.row_height,
        top=0.905,
        main_title_font=max(tier.panel_font + 2.6, 10.5),
        panel_font=tier.panel_font,
        font=tier.font,
        label_font=tier.font + 0.6,
        legend_font=max(tier.font + 0.7, 6.2),
        legend_marker=max(tier.marker_recent * 0.75, 4.2),
        marker_ts=tier.marker_ts,
        marker_recent=tier.marker_recent,
        line_width=max(tier.marker_ts / 8, 0.28),
        heatmap_years=tier.heatmap_years,
        row_hspace=min(tier.row_hspace + 0.055, 0.24),
        inner_hspace=0.54,
        inner_wspace=0.09,
        bottom_ratio=tier.bottom_ratio,
    )


def phase_legend_handles(marker_size: float) -> List[Line2D]:
    """One coloured dot per phase, in `PHASES` order."""
    return [Line2D([0], [0], marker="o", color="w", markerfacecolor=PHASE_COLORS[p],
                   markersize=marker_size, label=p) for p in PHASES]


# ---- Axes-level drawing helpers ---------------------------------------------
def style_axes(ax, layout: RiverLayout) -> None:
    """Compact tick/grid style shared by the line and seasonality panels."""
    ax.tick_params(axis="both", labelsize=layout.font, pad=1.0, width=0.45, length=2.0)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    for spine in ax.spines.values():
        spine.set_linewidth(0.45)
    ax.grid(alpha=0.20, linewidth=0.45)


def scatter_by_phase(ax, df: pd.DataFrame, size: float, alpha: float) -> None:
    """Grey level line with the observations coloured by phase on top."""
    for phase in PHASES:
        mask = df["phase"] == phase
        if mask.any():
            ax.scatter(df.loc[mask, "date"], df.loc[mask, "level"], c=PHASE_COLORS[phase],
                       s=size, alpha=alpha, zorder=2, rasterized=True)


def month_tick_labels(layout: RiverLayout) -> List[str]:
    """Month initials; every second one is hidden when the font is tiny."""
    return MONTH_LETTERS_SPARSE if layout.font <= 5.2 else MONTH_LETTERS


def plot_full_series(ax, df, layout: RiverLayout) -> None:
    ax.plot(df["date"], df["level"], color="gray", lw=layout.line_width, alpha=0.35, zorder=1)
    scatter_by_phase(ax, df, layout.marker_ts, alpha=0.78)
    ax.set_xlim(df["date"].min(), df["date"].max())
    ax.set_ylabel("m", fontsize=layout.font, labelpad=1)
    # Tick spacing grows with the length of the record.
    n_years = df["date"].dt.year.max() - df["date"].dt.year.min()
    base = 20 if n_years > 50 else 10 if n_years > 25 else 5 if n_years > 10 else 2
    ax.xaxis.set_major_locator(mdates.YearLocator(base=base))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    style_axes(ax, layout)


def plot_recent_period(ax, df, layout: RiverLayout, cfg: Config) -> None:
    start = pd.Timestamp(cfg.recent_start)
    recent = df[df["date"] >= start]
    if recent.empty:
        ax.set_axis_off()
        return
    ax.plot(recent["date"], recent["level"], color="gray", lw=layout.line_width, alpha=0.35, zorder=1)
    scatter_by_phase(ax, recent, layout.marker_recent, alpha=0.88)
    ax.set_xlim(start, recent["date"].max())
    ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    style_axes(ax, layout)


def plot_seasonality_compact(ax, df, layout: RiverLayout) -> None:
    monthly = monthly_percentiles(df)
    ax.fill_between(monthly.index, monthly["p10"], monthly["p90"], color="#56B4E9", alpha=0.14)
    ax.fill_between(monthly.index, monthly["p25"], monthly["p75"], color="#56B4E9", alpha=0.34)
    ax.plot(monthly.index, monthly["p50"], "o-", color="k", lw=0.70, ms=max(layout.marker_ts * 0.55, 1.4))
    ax.set_xlim(1, 12)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_tick_labels(layout))
    style_axes(ax, layout)


def plot_phase_heatmap(ax, df, layout: RiverLayout) -> None:
    """Dominant phase of each month over the last `heatmap_years` years."""
    year = df["date"].dt.year
    max_year = int(year.max())
    min_year = max_year - layout.heatmap_years + 1
    recent = df[year.between(min_year, max_year)]

    # Most frequent phase per (year, month); ties resolve alphabetically.
    keys = [recent["date"].dt.year.rename("year"), recent["date"].dt.month.rename("month")]
    dominant = recent.groupby(keys)["phase"].agg(lambda x: x.mode()[0])
    grid = dominant.unstack().reindex(index=range(min_year, max_year + 1), columns=range(1, 13))
    code_of = {phase: i + 1 for i, phase in enumerate(PHASES)}          # 0 = no data
    codes = grid.apply(lambda col: col.map(code_of)).fillna(0).astype(float)

    cmap = ListedColormap(["#F3F4F6"] + [PHASE_COLORS[p] for p in PHASES])
    ax.imshow(codes, aspect="auto", cmap=cmap, vmin=0, vmax=4, interpolation="nearest")
    years = list(codes.index)
    if len(years) <= 3:
        ax.set_yticks(range(len(years)))
        ax.set_yticklabels(years)
    else:
        ax.set_yticks([0, len(years) - 1])
        ax.set_yticklabels([years[0], years[-1]])
    ax.set_xticks(range(12))
    ax.set_xticklabels(month_tick_labels(layout))
    ax.tick_params(axis="both", labelsize=layout.font, pad=1.0, width=0.45, length=2.0)
    for spine in ax.spines.values():
        spine.set_linewidth(0.45)


# ---- Figure assembly ---------------------------------------------------------
def draw_station_block(fig, outer_spec, st: Station, layout: RiverLayout, cfg: Config) -> None:
    """One station: a label column on the left and the four panels on the right."""
    outer = outer_spec.subgridspec(1, 2, width_ratios=[0.78, 9.20], wspace=0.035)

    ax_label = fig.add_subplot(outer[0, 0])
    ax_label.set_axis_off()
    label = st.code + (f"\n{st.region}" if st.region else "")
    ax_label.text(0.02, 0.52, label, ha="left", va="center", fontsize=layout.label_font,
                  fontweight="bold", color="#1f2933", transform=ax_label.transAxes)

    inner = outer[0, 1].subgridspec(
        3, 2, width_ratios=[2.45, 1.65], height_ratios=[0.12, 1.00, layout.bottom_ratio],
        hspace=layout.inner_hspace, wspace=layout.inner_wspace,
    )
    for column, title in [(0, "a) Complete series and heatmap"), (1, "b) Seasonality and recent years")]:
        ax_header = fig.add_subplot(inner[0, column])
        ax_header.set_axis_off()
        ax_header.text(0.0, 0.35, title, ha="left", va="center", fontsize=layout.panel_font,
                       fontweight="bold", color="#1f2933", transform=ax_header.transAxes)

    ax_series = fig.add_subplot(inner[1, 0])
    ax_heatmap = fig.add_subplot(inner[2, 0])
    ax_recent = fig.add_subplot(inner[1, 1])
    ax_season = fig.add_subplot(inner[2, 1])
    plot_full_series(ax_series, st.series, layout)
    plot_phase_heatmap(ax_heatmap, st.series, layout)
    plot_recent_period(ax_recent, st.series, layout, cfg)
    plot_seasonality_compact(ax_season, st.series, layout)


def plot_river_panels(stations: List[Station], cfg: Config, out_dir: Path) -> List[Path]:
    """Save one PNG per river with all of its stations. Returns the file paths."""
    if not stations:
        print("No valid stations for the river panels.")
        return []
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    for river, river_stations in group_by_river(stations).items():
        river_stations = sorted(river_stations, key=lambda st: st.code)
        layout = river_layout(river, len(river_stations), cfg)

        fig = plt.figure(figsize=(layout.fig_width, layout.fig_height), facecolor="white", constrained_layout=False)
        grid = fig.add_gridspec(nrows=len(river_stations), ncols=1, left=0.035, right=0.985,
                                top=layout.top, bottom=0.045, hspace=layout.row_hspace)
        fig.suptitle(f"Hydrological Analysis by River – {river}", fontsize=layout.main_title_font,
                     fontweight="bold", y=0.985)
        fig.legend(handles=phase_legend_handles(layout.legend_marker), loc="upper center", ncol=4,
                   bbox_to_anchor=(0.5, 0.942), frameon=False, fontsize=layout.legend_font,
                   handlelength=1.0, columnspacing=1.3)
        for row, st in enumerate(river_stations):
            draw_station_block(fig, grid[row, 0], st, layout, cfg)

        path = out_dir / f"hydrological_analysis_{safe_filename(cfg.dataset_label)}_{safe_filename(river)}_panel.png"
        fig.savefig(path, dpi=300, facecolor="white")
        print(f"  saved river panel: {path}")
        saved.append(path)
        if cfg.show_figures:
            plt.show()
        plt.close(fig)
    return saved


In [ ]:
# =============================================================================
# BLOCK 5 — FIGURE 2: SEASONALITY OF ALL STATIONS, GROUPED BY RIVER
# =============================================================================
# One compact figure: a vertical label per river and, next to it, a small
# seasonality chart per station (monthly P10-P90 and P25-P75 bands, monthly
# median line, and one dot per month coloured by its seasonal phase).

SEASONALITY_LEGEND_ORDER = ("High", "Falling", "Low", "Rising")


def plot_station_seasonality(ax, st: Station) -> None:
    """Monthly percentile bands and median for one station."""
    monthly = monthly_percentiles(st.series)
    title_lines = [st.code] + ([st.region] if st.region else [])
    ax.set_title("\n".join(title_lines), fontsize=7.4, fontweight="bold", pad=3.5)

    month_phase = assign_month_phases(monthly)
    ax.fill_between(monthly.index, monthly["p10"], monthly["p90"], color="#56B4E9", alpha=0.15, zorder=1)
    ax.fill_between(monthly.index, monthly["p25"], monthly["p75"], color="#56B4E9", alpha=0.35, zorder=2)
    ax.plot(monthly.index, monthly["p50"], "-", color="gray", lw=1.0, alpha=0.65, zorder=3)
    for month, median in monthly["p50"].items():
        ax.scatter(month, median, c=PHASE_COLORS.get(month_phase[month], "gray"), s=28,
                   edgecolor="white", linewidth=0.45, zorder=4)

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LETTERS, fontsize=7.2)
    ax.grid(alpha=0.3, linestyle="--")
    ax.tick_params(axis="y", labelsize=7.2, pad=1.4)


def seasonality_legend_handles(marker_size: float = 7) -> List[Line2D]:
    dots = [Line2D([0], [0], marker="o", color="w", markerfacecolor=PHASE_COLORS[p],
                   markersize=marker_size, markeredgecolor="white", label=p)
            for p in SEASONALITY_LEGEND_ORDER]
    return dots + [Line2D([0], [0], color="gray", lw=1.0, alpha=0.7, label="Monthly median")]


def plot_seasonality_all_rivers(stations: List[Station], cfg: Config, out_dir: Path,
                                fig_width: float = 8.27, panel_height: float = 2.05,
                                river_gap_height: float = 0.38):
    """Save the all-rivers seasonality figure (PNG + PDF). Returns the PNG path."""
    rivers = {river: sort_by_peak_month(group) for river, group in group_by_river(stations).items()}
    if not rivers:
        print("No river groups to plot.")
        return None

    n_cols = max(1, min(cfg.seasonality_max_cols, max(len(g) for g in rivers.values())))
    rows_per_river = {river: max(1, math.ceil(len(g) / n_cols)) for river, g in rivers.items()}
    fig_height = max(6.4, sum(rows_per_river.values()) * panel_height
                     + max(0, len(rivers) - 1) * river_gap_height + 1.20)

    fig = plt.figure(figsize=(fig_width, fig_height), facecolor="white", constrained_layout=False)
    outer = fig.add_gridspec(nrows=len(rivers), ncols=2, width_ratios=[0.16, 1.0],
                             height_ratios=[rows_per_river[r] for r in rivers],
                             left=0.055, right=0.995, top=0.985, bottom=0.072, hspace=0.34, wspace=0.025)

    for i, (river, group) in enumerate(rivers.items()):
        # Vertical river label on the left.
        ax_label = fig.add_subplot(outer[i, 0])
        ax_label.set_axis_off()
        ax_label.text(0.52, 0.5, river, rotation=90, ha="center", va="center", fontsize=9.0,
                      fontweight="bold", color="#1f2933",
                      bbox=dict(boxstyle="round,pad=0.22", facecolor="#F4F7FA", edgecolor="#D5DCE5", linewidth=0.7))

        # One small chart per station; unused grid cells are hidden.
        inner = outer[i, 1].subgridspec(rows_per_river[river], n_cols, hspace=0.42, wspace=0.24)
        for k in range(rows_per_river[river] * n_cols):
            ax = fig.add_subplot(inner[k // n_cols, k % n_cols])
            if k < len(group):
                plot_station_seasonality(ax, group[k])
            else:
                ax.set_visible(False)

    fig.supxlabel("Month", fontsize=9.2, y=0.035)
    fig.supylabel("Water level (m)", fontsize=9.2, x=0.010)
    fig.legend(handles=seasonality_legend_handles(), loc="lower center", ncol=5, bbox_to_anchor=(0.5, 0.002),
               frameon=False, fontsize=8.2, handlelength=1.3, columnspacing=0.95)

    out_dir.mkdir(parents=True, exist_ok=True)
    png_path = out_dir / "seasonality_all_rivers_compact.png"
    pdf_path = out_dir / "seasonality_all_rivers_compact.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", pad_inches=0.035, facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.035, facecolor="white")
    print(f"Saved: {png_path}\nSaved: {pdf_path}")
    if cfg.show_figures:
        plt.show()
    plt.close(fig)
    return png_path


In [ ]:
# =============================================================================
# BLOCK 6 — FIGURE 3: STATION SUMMARY CARDS, ONE FIGURE PER RIVER
# =============================================================================
# Each station is drawn as a small "card" listing its period of record, mean
# annual amplitude, number of observations and gaps longer than `gap_days`.


def draw_station_card(ax, summary: dict, cfg: Config) -> None:
    """Draw one summary card into an empty axes."""
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor="#b8c2cc", lw=0.75))
    ax.add_patch(plt.Rectangle((0, 0.79), 1, 0.21, facecolor="#243447", edgecolor="#243447", lw=0))

    region = summary[cfg.meta_region_col.lower()]
    title = f"{summary['code']} | {region}" if region else summary["code"]
    ax.text(0.035, 0.895, textwrap.fill(title, width=28), ha="left", va="center", fontsize=7.5,
            fontweight="bold", color="white")

    amplitude = summary["mean_annual_amplitude"]
    rows = [
        ("Period", summary["period"]),
        ("Mean annual amp.", f"{amplitude:.2f} m" if pd.notna(amplitude) else "NA"),
        ("Observations", f"{summary['n_obs']:,}"),
        (f"Gaps >{cfg.gap_days} d", str(summary["large_gaps"])),
        ("Longest gap", summary["longest_gap"]),
    ]
    y, row_height = 0.705, 0.127
    for i, (label, value) in enumerate(rows):
        background = "#f4f7fa" if i % 2 == 0 else "#ffffff"
        ax.add_patch(plt.Rectangle((0.02, y - row_height + 0.01), 0.96, row_height,
                                   facecolor=background, edgecolor="#d8dee6", lw=0.35))
        ax.text(0.045, y - 0.045, label, ha="left", va="center", fontsize=6.7, fontweight="bold", color="#334155")
        ax.text(0.42, y - 0.045, textwrap.fill(str(value), width=27), ha="left", va="center",
                fontsize=6.45, color="#111827")
        y -= row_height


def plot_river_cards(summaries: List[dict], river: str, cfg: Config, out_dir: Path):
    """Save the card grid of one river (PNG + PDF)."""
    summaries = sorted(summaries, key=lambda s: s["code"])
    n = len(summaries)
    n_cols = min(cfg.station_cards_max_cols, n)
    n_rows = math.ceil(n / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(max(7.2, n_cols * 3.35), max(2.9, n_rows * 2.08 + 0.65)),
                             facecolor="white", squeeze=False)
    for k in range(n_rows * n_cols):
        ax = axes[k // n_cols, k % n_cols]
        if k < n:
            draw_station_card(ax, summaries[k], cfg)
        else:
            ax.set_visible(False)

    fig.suptitle(f"Station summary by river: {river}", fontsize=11, fontweight="bold", y=0.995, color="#1f2933")
    fig.tight_layout(rect=[0, 0, 1, 0.965], h_pad=0.5, w_pad=0.45)

    out_dir.mkdir(parents=True, exist_ok=True)
    stem = out_dir / f"station_summary_{safe_filename(river)}"
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    print(f"Saved: {stem.with_suffix('.png')}\nSaved: {stem.with_suffix('.pdf')}")
    if cfg.show_figures:
        plt.show()
    plt.close(fig)


def plot_station_summary_cards(stations: List[Station], cfg: Config, out_dir: Path) -> None:
    """One card figure per river plus a CSV with the numbers shown on the cards."""
    summaries = [station_summary(st, cfg) for st in stations]
    if not summaries:
        print("No valid stations to summarise.")
        return

    by_river: Dict[str, List[dict]] = {}
    for summary in summaries:
        by_river.setdefault(summary["river"], []).append(summary)
    for river, river_summaries in sorted(by_river.items()):
        print(f"River: {river} | stations: {len(river_summaries)}")
        plot_river_cards(river_summaries, river, cfg, out_dir)

    csv_path = out_dir / "station_summary_by_river_values.csv"
    pd.DataFrame(summaries).sort_values(["river", "code"]).to_csv(csv_path, index=False)
    print(f"Saved values table: {csv_path}")


In [ ]:
# =============================================================================
# BLOCK 7 — RUN
# =============================================================================
# Switch individual outputs on/off with the RUN_* flags, then run the cell.

RUN_PHASE_TABLES = True     # per-station CSVs with a `phase` column
RUN_RIVER_PANELS = True     # Figure 1: station panels, one figure per river
RUN_SEASONALITY = True      # Figure 2: seasonality of all stations
RUN_STATION_CARDS = True    # Figure 3: station summary cards + values CSV


def run_analysis(cfg: Config) -> None:
    metadata = load_metadata(cfg)
    series_by_code = load_all_series(cfg)

    # Classify every day of every station (Block 2).
    stations = [
        station_from_metadata(code, metadata, cfg, series=add_phase(series, cfg))
        for code, series in series_by_code.items()
    ]

    if RUN_PHASE_TABLES:
        write_phase_tables(stations, cfg)

    out_dir = Path(cfg.output_dir)
    if RUN_RIVER_PANELS:
        print("\nFigure 1 — river panels")
        plot_river_panels(stations, cfg, out_dir / "hydrological_analysis_by_river")

    # Figures 2 and 3 only include stations listed in the metadata table,
    # in the order of that table.
    if metadata.empty:
        print("\nFigures 2 and 3 need a metadata table; skipped.")
        return
    listed = [
        station_from_metadata(code, metadata, cfg, series=series_by_code[code])
        for code in metadata[cfg.meta_code_col] if code in series_by_code
    ]
    if RUN_SEASONALITY:
        print("\nFigure 2 — seasonality")
        plot_seasonality_all_rivers(listed, cfg, out_dir / "seasonality_by_river")
    if RUN_STATION_CARDS:
        print("\nFigure 3 — station summary cards")
        plot_station_summary_cards(listed, cfg, out_dir / "station_summaries_by_river")


run_analysis(CFG)
